In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Working with Joins") \
    .getOrCreate()

In [2]:
customers = [

(1, "Rahul", "Hyderabad"),
(2, "Priya", "Bangalore"),
(3, "Amit", "Mumbai"),
(4, "Sneha", "Chennai"),
(5, "Farhan", "Delhi")

]

customer_columns = [
    "customer_id",
    "customer_name",
    "city"
]

customers_df = spark.createDataFrame(
    customers,
    customer_columns
)

customers_df.show()


orders = [

(101, 1, "Laptop", 65000),
(102, 2, "Mobile", 25000),
(103, 1, "TV", 45000),
(104, 3, "Chair", 5000),
(105, 7, "Watch", 8000)

]

order_columns = [
    "order_id",
    "customer_id",
    "product",
    "amount"
]

orders_df = spark.createDataFrame(
    orders,
    order_columns
)

orders_df.show()

+-----------+-------------+---------+
|customer_id|customer_name|     city|
+-----------+-------------+---------+
|          1|        Rahul|Hyderabad|
|          2|        Priya|Bangalore|
|          3|         Amit|   Mumbai|
|          4|        Sneha|  Chennai|
|          5|       Farhan|    Delhi|
+-----------+-------------+---------+

+--------+-----------+-------+------+
|order_id|customer_id|product|amount|
+--------+-----------+-------+------+
|     101|          1| Laptop| 65000|
|     102|          2| Mobile| 25000|
|     103|          1|     TV| 45000|
|     104|          3|  Chair|  5000|
|     105|          7|  Watch|  8000|
+--------+-----------+-------+------+



In [3]:
customers_df.join(
    orders_df,
    "customer_id",
    "inner"
).show()

+-----------+-------------+---------+--------+-------+------+
|customer_id|customer_name|     city|order_id|product|amount|
+-----------+-------------+---------+--------+-------+------+
|          1|        Rahul|Hyderabad|     101| Laptop| 65000|
|          1|        Rahul|Hyderabad|     103|     TV| 45000|
|          2|        Priya|Bangalore|     102| Mobile| 25000|
|          3|         Amit|   Mumbai|     104|  Chair|  5000|
+-----------+-------------+---------+--------+-------+------+



In [4]:
customers_df.join(
    orders_df,
    "customer_id",
    "left"
).show()

+-----------+-------------+---------+--------+-------+------+
|customer_id|customer_name|     city|order_id|product|amount|
+-----------+-------------+---------+--------+-------+------+
|          1|        Rahul|Hyderabad|     103|     TV| 45000|
|          1|        Rahul|Hyderabad|     101| Laptop| 65000|
|          2|        Priya|Bangalore|     102| Mobile| 25000|
|          5|       Farhan|    Delhi|    NULL|   NULL|  NULL|
|          3|         Amit|   Mumbai|     104|  Chair|  5000|
|          4|        Sneha|  Chennai|    NULL|   NULL|  NULL|
+-----------+-------------+---------+--------+-------+------+



In [5]:
customers_df.join(
    orders_df,
    "customer_id",
    "right"
).show()

+-----------+-------------+---------+--------+-------+------+
|customer_id|customer_name|     city|order_id|product|amount|
+-----------+-------------+---------+--------+-------+------+
|          1|        Rahul|Hyderabad|     101| Laptop| 65000|
|          2|        Priya|Bangalore|     102| Mobile| 25000|
|          7|         NULL|     NULL|     105|  Watch|  8000|
|          1|        Rahul|Hyderabad|     103|     TV| 45000|
|          3|         Amit|   Mumbai|     104|  Chair|  5000|
+-----------+-------------+---------+--------+-------+------+



In [6]:
from pyspark.sql.functions import sum

customers_df.join(
    orders_df,
    "customer_id"
).groupBy(
    "customer_name"
).agg(
    sum("amount").alias("total_spent")
).show()

+-------------+-----------+
|customer_name|total_spent|
+-------------+-----------+
|        Priya|      25000|
|        Rahul|     110000|
|         Amit|       5000|
+-------------+-----------+



In [7]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *


In [14]:
employees = [

(101,"Rahul","IT",75000),
(102,"Priya","IT",85000),
(103,"Amit","IT",65000),

(104,"Sneha","HR",70000),
(105,"Farhan","HR",90000),

(106,"Neha","Finance",95000),
(107,"Arjun","Finance",80000),
(108,"Meera","Finance",75000)

]

In [15]:
columns = ["employee_id", "employee_name", "department", "salary"]

df = spark.createDataFrame(employees, columns)

In [16]:
window_spec = Window.orderBy(
    col("salary").desc()
)
df.withColumn(
    "row_number",
    row_number().over(window_spec)
).show()

+-----------+-------------+----------+------+----------+
|employee_id|employee_name|department|salary|row_number|
+-----------+-------------+----------+------+----------+
|        106|         Neha|   Finance| 95000|         1|
|        105|       Farhan|        HR| 90000|         2|
|        102|        Priya|        IT| 85000|         3|
|        107|        Arjun|   Finance| 80000|         4|
|        101|        Rahul|        IT| 75000|         5|
|        108|        Meera|   Finance| 75000|         6|
|        104|        Sneha|        HR| 70000|         7|
|        103|         Amit|        IT| 65000|         8|
+-----------+-------------+----------+------+----------+



In [17]:
window_spec = Window.orderBy(
    col("salary").desc()
)

df.withColumn(
    "rank",
    rank().over(window_spec)
).show()

+-----------+-------------+----------+------+----+
|employee_id|employee_name|department|salary|rank|
+-----------+-------------+----------+------+----+
|        106|         Neha|   Finance| 95000|   1|
|        105|       Farhan|        HR| 90000|   2|
|        102|        Priya|        IT| 85000|   3|
|        107|        Arjun|   Finance| 80000|   4|
|        101|        Rahul|        IT| 75000|   5|
|        108|        Meera|   Finance| 75000|   5|
|        104|        Sneha|        HR| 70000|   7|
|        103|         Amit|        IT| 65000|   8|
+-----------+-------------+----------+------+----+



In [18]:
window_spec = Window.orderBy(
    col("salary").desc()
)

df.withColumn(
    "dense_rank",
    dense_rank().over(window_spec)
).show()

+-----------+-------------+----------+------+----------+
|employee_id|employee_name|department|salary|dense_rank|
+-----------+-------------+----------+------+----------+
|        106|         Neha|   Finance| 95000|         1|
|        105|       Farhan|        HR| 90000|         2|
|        102|        Priya|        IT| 85000|         3|
|        107|        Arjun|   Finance| 80000|         4|
|        101|        Rahul|        IT| 75000|         5|
|        108|        Meera|   Finance| 75000|         5|
|        104|        Sneha|        HR| 70000|         6|
|        103|         Amit|        IT| 65000|         7|
+-----------+-------------+----------+------+----------+



In [19]:
window_spec = Window.partitionBy(
    "department"
).orderBy(
    col("salary").desc()
)

df.withColumn(
    "department_rank",
    rank().over(window_spec)
).show()

+-----------+-------------+----------+------+---------------+
|employee_id|employee_name|department|salary|department_rank|
+-----------+-------------+----------+------+---------------+
|        106|         Neha|   Finance| 95000|              1|
|        107|        Arjun|   Finance| 80000|              2|
|        108|        Meera|   Finance| 75000|              3|
|        105|       Farhan|        HR| 90000|              1|
|        104|        Sneha|        HR| 70000|              2|
|        102|        Priya|        IT| 85000|              1|
|        101|        Rahul|        IT| 75000|              2|
|        103|         Amit|        IT| 65000|              3|
+-----------+-------------+----------+------+---------------+



In [20]:
# EXERCISE:
%%writefile patients.csv
patient_id,patient_name,city,age,gender,blood_group,insurance_status
101,Rahul Sharma,Hyderabad,35,Male,O+,Active
102,Priya Reddy,Bangalore,29,Female,A+,Active
103,Amit Kumar,Mumbai,42,Male,B+,Inactive
104,Sneha Patel,Chennai,31,Female,O+,Active
105,Farhan Ali,Delhi,55,Male,AB+,Active
106,Neha Singh,,38,Female,A+,Inactive
107,Arjun Verma,Pune,26,Male,B+,Active
108,Meera Nair,Kochi,48,Female,O-,Active
109,Kiran Rao,Hyderabad,33,Male,,Inactive
110,Nisha Reddy,Bangalore,41,Female,A+,Active

Writing patients.csv


In [43]:
%%writefile appointments.csv
appointment_id,patient_id,doctor_name,department,appointment_date,consultation_fee,status
5001,101,Dr. Ramesh,Cardiology,2025-01-10,1500,Completed
5002,102,Dr. Suresh,Neurology,2025-01-11,2000,Completed
5003,101,Dr. Anita,Dermatology,2025-01-15,1000,Completed
5004,103,Dr. Ramesh,Cardiology,2025-01-20,1500,Cancelled
5005,104,Dr. Priya,Orthopedics,2025-01-22,2500,Completed
5006,105,Dr. Anita,Dermatology,2025-01-25,1000,Pending
5007,107,Dr. Suresh,Neurology,2025-02-01,2000,Completed
5008,110,Dr. Priya,Orthopedics,2025-02-03,2500,Completed
5009,120,Dr. Ramesh,Cardiology,2025-02-05,1500,Completed
5010,108,Dr. Anita,Dermatology,2025-02-10,,Pending

Overwriting appointments.csv


In [22]:
%%writefile patient_preferences.json
[
{
"patient_id":101,
"preferred_hospital":"Apollo",
"contact":{
"phone":"9876500011",
"email":"rahul@gmail.com"
}
},
{
"patient_id":102,
"preferred_hospital":"Yashoda",
"contact":{
"phone":null,
"email":"priya@gmail.com"
}
},
{
"patient_id":103,
"preferred_hospital":"Care",
"contact":{
"phone":"9876500013",
"email":null
}
},
{
"patient_id":104,
"preferred_hospital":null,
"contact":{
"phone":"9876500014",
"email":"sneha@gmail.com"
}

}
]

Writing patient_preferences.json


In [23]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Healthcare Analytics") \
    .getOrCreate()

In [24]:
# PART 1: CSV INGESTION
patients_df = spark.read.csv(
    "patients.csv",
    header=True,
    inferSchema=True
)

In [44]:
appointments_df = spark.read.csv(
    "appointments.csv",
    header=True,
    inferSchema=True
)

In [26]:
preferences_df = spark.read.json(
    "patient_preferences.json"
)

In [27]:
patients_df.printSchema()
appointments_df.printSchema()

root
 |-- patient_id: integer (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- blood_group: string (nullable = true)
 |-- insurance_status: string (nullable = true)

root
 |-- appointment_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- appointment_date: timestamp (nullable = true)
 |-- consult: integer (nullable = true)



In [28]:
patients_df.count()
appointments_df.count()

10

In [31]:
patients_df.show(5)
appointments_df.show(5)

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
+----------+------------+---------+---+------+-----------+----------------+
only showing top 5 rows
+--------------+----------+-----------+-----------+-------------------+-------+
|appointment_id|patient_id|doctor_name| department|   appointment_date|consult|
+--------------+----------+-----------+-----------+-------------------+-------+
|          5001|       101| Dr. Ramesh| Cardiology|2

In [32]:
patients_df.select("city").distinct().show()

+---------+
|     city|
+---------+
|Bangalore|
|    Kochi|
|  Chennai|
|   Mumbai|
|     Pune|
|    Delhi|
|Hyderabad|
|     NULL|
+---------+



In [33]:
appointments_df.select("department").distinct().show()

+-----------+
| department|
+-----------+
|  Neurology|
|Dermatology|
| Cardiology|
|Orthopedics|
+-----------+



In [35]:
patients_df.write.mode("overwrite").parquet("patients_parquet")

In [36]:
patients_parquet_df = spark.read.parquet("patients_parquet")

In [37]:
print("CSV Count:", patients_df.count())
print("Parquet Count:", patients_parquet_df.count())

CSV Count: 10
Parquet Count: 10


In [38]:
# PART 2: FILTERING:
patients_df.filter(
    patients_df.city == "Hyderabad"
).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [39]:
patients_df.filter(
    patients_df.gender == "Female"
).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [40]:
patients_df.filter(
    patients_df.age > 40
).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [45]:
appointments_df.filter(
    appointments_df.status == "Completed"
).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|      2025-02-03|            2500|Completed|
|          5009|       120| Dr. Ramesh| Cardiology|      2025-02-05|            1500|Completed|
+--------------+----------+-----------+-

In [46]:
appointments_df.filter(
    appointments_df.status == "Pending"
).show()

+--------------+----------+-----------+-----------+----------------+----------------+-------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee| status|
+--------------+----------+-----------+-----------+----------------+----------------+-------+
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|Pending|
|          5010|       108|  Dr. Anita|Dermatology|      2025-02-10|            NULL|Pending|
+--------------+----------+-----------+-----------+----------------+----------------+-------+



In [47]:
appointments_df.filter(
    appointments_df.consultation_fee > 1500
).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|      2025-02-03|            2500|Completed|
+--------------+----------+-----------+-----------+----------------+----------------+---------+



In [48]:
patients_df.filter(
    patients_df.insurance_status == "Active"
).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [49]:
patients_df.filter(
    patients_df.insurance_status == "Inactive"
).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [50]:
appointments_df.filter(
    appointments_df.department == "Cardiology"
).show()

+--------------+----------+-----------+----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name|department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh|Cardiology|      2025-01-10|            1500|Completed|
|          5004|       103| Dr. Ramesh|Cardiology|      2025-01-20|            1500|Cancelled|
|          5009|       120| Dr. Ramesh|Cardiology|      2025-02-05|            1500|Completed|
+--------------+----------+-----------+----------+----------------+----------------+---------+



In [51]:
# PART 3: NULL HANDLING
patients_df.filter(
    patients_df.city.isNull()
).show()

+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|       106|  Neha Singh|NULL| 38|Female|         A+|        Inactive|
+----------+------------+----+---+------+-----------+----------------+



In [52]:
patients_df.filter(
    patients_df.blood_group.isNull()
).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [53]:
appointments_df.filter(
    appointments_df.consultation_fee.isNull()
).show()

+--------------+----------+-----------+-----------+----------------+----------------+-------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee| status|
+--------------+----------+-----------+-----------+----------------+----------------+-------+
|          5010|       108|  Dr. Anita|Dermatology|      2025-02-10|            NULL|Pending|
+--------------+----------+-----------+-----------+----------------+----------------+-------+



In [54]:
from pyspark.sql.functions import col, sum

patients_df.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in patients_df.columns]
).show()

appointments_df.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in appointments_df.columns]
).show()

+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|         0|           0|   1|  0|     0|          1|               0|
+----------+------------+----+---+------+-----------+----------------+

+--------------+----------+-----------+----------+----------------+----------------+------+
|appointment_id|patient_id|doctor_name|department|appointment_date|consultation_fee|status|
+--------------+----------+-----------+----------+----------------+----------------+------+
|             0|         0|          0|         0|               0|               1|     0|
+--------------+----------+-----------+----------+----------------+----------------+------+



In [65]:
patients_df = patients_df.fillna(
    {"city": "Unknown"}
)

In [62]:
print(patients_df)

None


In [66]:
patients_df = spark.read.csv(
    "patients.csv",
    header=True,
    inferSchema=True
)

In [67]:
patients_df = patients_df.fillna(
    {"blood_group": "Not Available"}
)

In [68]:
appointments_df = appointments_df.fillna(
    {"consultation_fee": 0}
)

In [69]:
appointments_no_null_fee_df = appointments_df.dropna(
    subset=["consultation_fee"]
)

In [70]:
from pyspark.sql.functions import when

patients_df = patients_df.withColumn(
    "data_quality_status",
    when(
        col("city").isNull() |
        col("blood_group").isNull(),
        "Incomplete"
    ).otherwise("Complete")
)

In [71]:
patients_df.groupBy(
    "data_quality_status"
).count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|    9|
|         Incomplete|    1|
+-------------------+-----+



In [72]:
# PART 4: BUILT-IN FUNCTIONS
from pyspark.sql.functions import upper

patients_df.select(
    upper("patient_name").alias("patient_name")
).show()

+------------+
|patient_name|
+------------+
|RAHUL SHARMA|
| PRIYA REDDY|
|  AMIT KUMAR|
| SNEHA PATEL|
|  FARHAN ALI|
|  NEHA SINGH|
| ARJUN VERMA|
|  MEERA NAIR|
|   KIRAN RAO|
| NISHA REDDY|
+------------+



In [73]:
from pyspark.sql.functions import lower

patients_df.select(
    lower("patient_name").alias("patient_name")
).show()

+------------+
|patient_name|
+------------+
|rahul sharma|
| priya reddy|
|  amit kumar|
| sneha patel|
|  farhan ali|
|  neha singh|
| arjun verma|
|  meera nair|
|   kiran rao|
| nisha reddy|
+------------+



In [74]:
from pyspark.sql.functions import length

patients_df.select(
    "patient_name",
    length("patient_name").alias("name_length")
).show()

+------------+-----------+
|patient_name|name_length|
+------------+-----------+
|Rahul Sharma|         12|
| Priya Reddy|         11|
|  Amit Kumar|         10|
| Sneha Patel|         11|
|  Farhan Ali|         10|
|  Neha Singh|         10|
| Arjun Verma|         11|
|  Meera Nair|         10|
|   Kiran Rao|          9|
| Nisha Reddy|         11|
+------------+-----------+



In [75]:
from pyspark.sql.functions import substring

patients_df.select(
    "patient_name",
    substring("patient_name", 1, 3).alias("first_3_letters")
).show()

+------------+---------------+
|patient_name|first_3_letters|
+------------+---------------+
|Rahul Sharma|            Rah|
| Priya Reddy|            Pri|
|  Amit Kumar|            Ami|
| Sneha Patel|            Sne|
|  Farhan Ali|            Far|
|  Neha Singh|            Neh|
| Arjun Verma|            Arj|
|  Meera Nair|            Mee|
|   Kiran Rao|            Kir|
| Nisha Reddy|            Nis|
+------------+---------------+



In [76]:
from pyspark.sql.functions import when

patients_df.withColumn(
    "age_group",
    when(patients_df.age < 30, "Young")
    .when(patients_df.age < 50, "Adult")
    .otherwise("Senior")
).show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|    Adult|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|    Young|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|    Adult|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|    Adult|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|           Complete|   Senior|
|       106|  Neha Singh|     NULL| 38|Female|           A+|        Inactive|         Incomplete|    Adult|
|       107| Arjun Verma|   

In [77]:
patients_df.withColumn(
    "insurance_flag",
    when(col("insurance_status") == "Active", 1)
    .otherwise(0)
).show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|insurance_flag|
+----------+------------+---------+---+------+-------------+----------------+-------------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|             1|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|             1|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|             0|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|             1|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|           Complete|             1|
|       106|  Neha Singh|     NULL| 38|Female|           A+|        Inactive|         Incomplete

In [78]:
patients_df.withColumn(
    "senior_citizen",
    when(col("age") >= 60, "Yes")
    .otherwise("No")
).show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|senior_citizen|
+----------+------------+---------+---+------+-------------+----------------+-------------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|            No|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|            No|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|            No|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|            No|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|           Complete|            No|
|       106|  Neha Singh|     NULL| 38|Female|           A+|        Inactive|         Incomplete

In [79]:
patients_df.select(
    concat_ws(" - ", "patient_name", "city")
    .alias("patient_city")
).show()

+--------------------+
|        patient_city|
+--------------------+
|Rahul Sharma - Hy...|
|Priya Reddy - Ban...|
| Amit Kumar - Mumbai|
|Sneha Patel - Che...|
|  Farhan Ali - Delhi|
|          Neha Singh|
|  Arjun Verma - Pune|
|  Meera Nair - Kochi|
|Kiran Rao - Hyder...|
|Nisha Reddy - Ban...|
+--------------------+



In [80]:
patients_df.select(
    trim("patient_name")
    .alias("patient_name")
).show()

+------------+
|patient_name|
+------------+
|Rahul Sharma|
| Priya Reddy|
|  Amit Kumar|
| Sneha Patel|
|  Farhan Ali|
|  Neha Singh|
| Arjun Verma|
|  Meera Nair|
|   Kiran Rao|
| Nisha Reddy|
+------------+



In [81]:
patients_df.withColumn(
    "city",
    upper("city")
).show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|
+----------+------------+---------+---+------+-------------+----------------+-------------------+
|       101|Rahul Sharma|HYDERABAD| 35|  Male|           O+|          Active|           Complete|
|       102| Priya Reddy|BANGALORE| 29|Female|           A+|          Active|           Complete|
|       103|  Amit Kumar|   MUMBAI| 42|  Male|           B+|        Inactive|           Complete|
|       104| Sneha Patel|  CHENNAI| 31|Female|           O+|          Active|           Complete|
|       105|  Farhan Ali|    DELHI| 55|  Male|          AB+|          Active|           Complete|
|       106|  Neha Singh|     NULL| 38|Female|           A+|        Inactive|         Incomplete|
|       107| Arjun Verma|     PUNE| 26|  Male|           B+|          Active|           Complete|
|       108|  Meera 

In [82]:
# PART 5: GROUP BY AND AGGREGATIONS
patients_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|     NULL|    1|
|   Mumbai|    1|
|     Pune|    1|
|    Delhi|    1|
|Hyderabad|    2|
+---------+-----+



In [83]:
patients_df.groupBy("gender").count().show()

+------+-----+
|gender|count|
+------+-----+
|Female|    5|
|  Male|    5|
+------+-----+



In [84]:
patients_df.groupBy("blood_group").count().show()

+-------------+-----+
|  blood_group|count|
+-------------+-----+
|          AB+|    1|
|           O+|    2|
|           O-|    1|
|           B+|    2|
|           A+|    3|
|Not Available|    1|
+-------------+-----+



In [85]:
appointments_df.groupBy("department").count().show()

+-----------+-----+
| department|count|
+-----------+-----+
|  Neurology|    2|
|Dermatology|    3|
| Cardiology|    3|
|Orthopedics|    2|
+-----------+-----+



In [86]:
patients_df.groupBy("city").agg(
    avg("age").alias("average_age")
).show()

+---------+-----------+
|     city|average_age|
+---------+-----------+
|Bangalore|       35.0|
|    Kochi|       48.0|
|  Chennai|       31.0|
|     NULL|       38.0|
|   Mumbai|       42.0|
|     Pune|       26.0|
|    Delhi|       55.0|
|Hyderabad|       34.0|
+---------+-----------+



In [87]:
patients_df.groupBy("city").agg(
    max("age").alias("maximum_age")
).show()

+---------+-----------+
|     city|maximum_age|
+---------+-----------+
|Bangalore|         41|
|    Kochi|         48|
|  Chennai|         31|
|     NULL|         38|
|   Mumbai|         42|
|     Pune|         26|
|    Delhi|         55|
|Hyderabad|         35|
+---------+-----------+



In [88]:
patients_df.groupBy("city").agg(
    min("age").alias("minimum_age")
).show()

+---------+-----------+
|     city|minimum_age|
+---------+-----------+
|Bangalore|         29|
|    Kochi|         48|
|  Chennai|         31|
|     NULL|         38|
|   Mumbai|         42|
|     Pune|         26|
|    Delhi|         55|
|Hyderabad|         33|
+---------+-----------+



In [89]:
appointments_df.groupBy("department").agg(
    sum("consultation_fee").alias("total_fee")
).show()

+-----------+---------+
| department|total_fee|
+-----------+---------+
|  Neurology|     4000|
|Dermatology|     2000|
| Cardiology|     4500|
|Orthopedics|     5000|
+-----------+---------+



In [90]:
appointments_df.groupBy("department").agg(
    sum("consultation_fee").alias("total_revenue")
).orderBy(
    col("total_revenue").desc()
).show(1)

+-----------+-------------+
| department|total_revenue|
+-----------+-------------+
|Orthopedics|         5000|
+-----------+-------------+
only showing top 1 row


In [91]:
# PART 6: JOINS
patients_df.join(
    appointments_df,
    "patient_id",
    "inner"
).show()

+----------+------------+---------+---+------+-----------+----------------+-------------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+---+------+-----------+----------------+-------------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|           Complete|          5002| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|          

In [92]:
patients_df.join(
    appointments_df,
    "patient_id",
    "left"
).show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+---+------+-------------+----------------+-------------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complet

In [93]:
patients_df.join(
    appointments_df,
    "patient_id",
    "right"
).show()

+----------+------------+---------+----+------+-----------+----------------+-------------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city| age|gender|blood_group|insurance_status|data_quality_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+----+------+-----------+----------------+-------------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad|  35|  Male|         O+|          Active|           Complete|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       102| Priya Reddy|Bangalore|  29|Female|         A+|          Active|           Complete|          5002| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|       101|Rahul Sharma|Hyderabad|  35|  Male|         O+|          Active|           Complete|    

In [94]:
patients_df.join(
    appointments_df,
    "patient_id",
    "full"
).show()

+----------+------------+---------+----+------+-------------+----------------+-------------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city| age|gender|  blood_group|insurance_status|data_quality_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+----+------+-------------+----------------+-------------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad|  35|  Male|           O+|          Active|           Complete|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       101|Rahul Sharma|Hyderabad|  35|  Male|           O+|          Active|           Complete|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|       102| Priya Reddy|Bangalore|  29|Female|           A+|          Active|           C

In [95]:
patients_df.join(
    appointments_df,
    "patient_id",
    "left"
).filter(
    appointments_df.patient_id.isNull()
).show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+--------------+-----------+----------+----------------+----------------+------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|appointment_id|doctor_name|department|appointment_date|consultation_fee|status|
+----------+------------+---------+---+------+-------------+----------------+-------------------+--------------+-----------+----------+----------------+----------------+------+
|       106|  Neha Singh|     NULL| 38|Female|           A+|        Inactive|         Incomplete|          NULL|       NULL|      NULL|            NULL|            NULL|  NULL|
|       109|   Kiran Rao|Hyderabad| 33|  Male|Not Available|        Inactive|           Complete|          NULL|       NULL|      NULL|            NULL|            NULL|  NULL|
+----------+------------+---------+---+------+-------------+----------------+-------------------+--------------+---

In [96]:
appointments_df.join(
    patients_df,
    "patient_id",
    "left"
).filter(
    patients_df.patient_id.isNull()
).show()

+----------+--------------+-----------+----------+----------------+----------------+---------+------------+----+----+------+-----------+----------------+-------------------+
|patient_id|appointment_id|doctor_name|department|appointment_date|consultation_fee|   status|patient_name|city| age|gender|blood_group|insurance_status|data_quality_status|
+----------+--------------+-----------+----------+----------------+----------------+---------+------------+----+----+------+-----------+----------------+-------------------+
|       120|          5009| Dr. Ramesh|Cardiology|      2025-02-05|            1500|Completed|        NULL|NULL|NULL|  NULL|       NULL|            NULL|               NULL|
+----------+--------------+-----------+----------+----------------+----------------+---------+------------+----+----+------+-----------+----------------+-------------------+



In [97]:
appointments_df.groupBy(
    "patient_id"
).count().show()

+----------+-----+
|patient_id|count|
+----------+-----+
|       108|    1|
|       101|    2|
|       103|    1|
|       120|    1|
|       107|    1|
|       102|    1|
|       105|    1|
|       110|    1|
|       104|    1|
+----------+-----+



In [98]:
patients_df.join(
    appointments_df,
    "patient_id"
).groupBy(
    "patient_name"
).agg(
    sum("consultation_fee").alias("total_fees")
).show()

+------------+----------+
|patient_name|total_fees|
+------------+----------+
|  Amit Kumar|      1500|
|Rahul Sharma|      2500|
|  Meera Nair|         0|
| Sneha Patel|      2500|
|  Farhan Ali|      1000|
| Arjun Verma|      2000|
| Nisha Reddy|      2500|
| Priya Reddy|      2000|
+------------+----------+



In [99]:
patients_df.join(
    appointments_df,
    "patient_id"
).groupBy(
    "patient_name"
).agg(
    sum("consultation_fee").alias("total_fees")
).show()

+------------+----------+
|patient_name|total_fees|
+------------+----------+
|  Amit Kumar|      1500|
|Rahul Sharma|      2500|
|  Meera Nair|         0|
| Sneha Patel|      2500|
|  Farhan Ali|      1000|
| Arjun Verma|      2000|
| Nisha Reddy|      2500|
| Priya Reddy|      2000|
+------------+----------+



In [100]:
patients_df.join(
    appointments_df,
    "patient_id"
).groupBy(
    "patient_name"
).agg(
    sum("consultation_fee").alias("total_fees")
).orderBy(
    col("total_fees").desc()
).show(1)

+------------+----------+
|patient_name|total_fees|
+------------+----------+
| Sneha Patel|      2500|
+------------+----------+
only showing top 1 row


In [101]:
patients_df.join(
    appointments_df,
    "patient_id"
).groupBy(
    "patient_name"
).agg(
    count("appointment_id").alias("appointment_count")
).show()

+------------+-----------------+
|patient_name|appointment_count|
+------------+-----------------+
|  Amit Kumar|                1|
|Rahul Sharma|                2|
|  Meera Nair|                1|
| Sneha Patel|                1|
|  Farhan Ali|                1|
| Arjun Verma|                1|
| Nisha Reddy|                1|
| Priya Reddy|                1|
+------------+-----------------+



In [102]:
# PART 7: WINDOW FUNCTIONS
from pyspark.sql.window import Window
from pyspark.sql.functions import *


In [103]:
patient_fees_df = patients_df.join(
    appointments_df,
    "patient_id"
).groupBy(
    "patient_id",
    "patient_name",
    "city"
).agg(
    sum("consultation_fee").alias("total_fees")
)

window_spec = Window.orderBy(
    col("total_fees").desc()
)

patient_fees_df.withColumn(
    "rank",
    rank().over(window_spec)
).show()

+----------+------------+---------+----------+----+
|patient_id|patient_name|     city|total_fees|rank|
+----------+------------+---------+----------+----+
|       110| Nisha Reddy|Bangalore|      2500|   1|
|       101|Rahul Sharma|Hyderabad|      2500|   1|
|       104| Sneha Patel|  Chennai|      2500|   1|
|       102| Priya Reddy|Bangalore|      2000|   4|
|       107| Arjun Verma|     Pune|      2000|   4|
|       103|  Amit Kumar|   Mumbai|      1500|   6|
|       105|  Farhan Ali|    Delhi|      1000|   7|
|       108|  Meera Nair|    Kochi|         0|   8|
+----------+------------+---------+----------+----+



In [104]:
patient_fees_df.withColumn(
    "dense_rank",
    dense_rank().over(window_spec)
).show()

+----------+------------+---------+----------+----------+
|patient_id|patient_name|     city|total_fees|dense_rank|
+----------+------------+---------+----------+----------+
|       110| Nisha Reddy|Bangalore|      2500|         1|
|       101|Rahul Sharma|Hyderabad|      2500|         1|
|       104| Sneha Patel|  Chennai|      2500|         1|
|       102| Priya Reddy|Bangalore|      2000|         2|
|       107| Arjun Verma|     Pune|      2000|         2|
|       103|  Amit Kumar|   Mumbai|      1500|         3|
|       105|  Farhan Ali|    Delhi|      1000|         4|
|       108|  Meera Nair|    Kochi|         0|         5|
+----------+------------+---------+----------+----------+



In [105]:
patient_fees_df.withColumn(
    "row_number",
    row_number().over(window_spec)
).show()

+----------+------------+---------+----------+----------+
|patient_id|patient_name|     city|total_fees|row_number|
+----------+------------+---------+----------+----------+
|       110| Nisha Reddy|Bangalore|      2500|         1|
|       101|Rahul Sharma|Hyderabad|      2500|         2|
|       104| Sneha Patel|  Chennai|      2500|         3|
|       102| Priya Reddy|Bangalore|      2000|         4|
|       107| Arjun Verma|     Pune|      2000|         5|
|       103|  Amit Kumar|   Mumbai|      1500|         6|
|       105|  Farhan Ali|    Delhi|      1000|         7|
|       108|  Meera Nair|    Kochi|         0|         8|
+----------+------------+---------+----------+----------+



In [106]:
patient_fees_df.orderBy(
    col("total_fees").desc()
).show(1)

+----------+------------+---------+----------+
|patient_id|patient_name|     city|total_fees|
+----------+------------+---------+----------+
|       101|Rahul Sharma|Hyderabad|      2500|
+----------+------------+---------+----------+
only showing top 1 row


In [107]:
patient_fees_df.orderBy(
    col("total_fees").desc()
).show(3)

+----------+------------+---------+----------+
|patient_id|patient_name|     city|total_fees|
+----------+------------+---------+----------+
|       110| Nisha Reddy|Bangalore|      2500|
|       101|Rahul Sharma|Hyderabad|      2500|
|       104| Sneha Patel|  Chennai|      2500|
+----------+------------+---------+----------+
only showing top 3 rows


In [108]:
city_window = Window.partitionBy(
    "city"
).orderBy(
    col("total_fees").desc()
)

patient_fees_df.withColumn(
    "city_rank",
    rank().over(city_window)
).filter(
    col("city_rank") == 1
).show()

+----------+------------+---------+----------+---------+
|patient_id|patient_name|     city|total_fees|city_rank|
+----------+------------+---------+----------+---------+
|       110| Nisha Reddy|Bangalore|      2500|        1|
|       104| Sneha Patel|  Chennai|      2500|        1|
|       105|  Farhan Ali|    Delhi|      1000|        1|
|       101|Rahul Sharma|Hyderabad|      2500|        1|
|       108|  Meera Nair|    Kochi|         0|        1|
|       103|  Amit Kumar|   Mumbai|      1500|        1|
|       107| Arjun Verma|     Pune|      2000|        1|
+----------+------------+---------+----------+---------+



In [109]:
city_window_low = Window.partitionBy(
    "city"
).orderBy(
    col("total_fees").asc()
)

patient_fees_df.withColumn(
    "city_rank",
    rank().over(city_window_low)
).filter(
    col("city_rank") == 1
).show()

+----------+------------+---------+----------+---------+
|patient_id|patient_name|     city|total_fees|city_rank|
+----------+------------+---------+----------+---------+
|       102| Priya Reddy|Bangalore|      2000|        1|
|       104| Sneha Patel|  Chennai|      2500|        1|
|       105|  Farhan Ali|    Delhi|      1000|        1|
|       101|Rahul Sharma|Hyderabad|      2500|        1|
|       108|  Meera Nair|    Kochi|         0|        1|
|       103|  Amit Kumar|   Mumbai|      1500|        1|
|       107| Arjun Verma|     Pune|      2000|        1|
+----------+------------+---------+----------+---------+



In [110]:
running_window = Window.orderBy(
    "patient_id"
).rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)

patient_fees_df.withColumn(
    "running_total",
    sum("total_fees").over(running_window)
).show()

+----------+------------+---------+----------+-------------+
|patient_id|patient_name|     city|total_fees|running_total|
+----------+------------+---------+----------+-------------+
|       101|Rahul Sharma|Hyderabad|      2500|         2500|
|       102| Priya Reddy|Bangalore|      2000|         4500|
|       103|  Amit Kumar|   Mumbai|      1500|         6000|
|       104| Sneha Patel|  Chennai|      2500|         8500|
|       105|  Farhan Ali|    Delhi|      1000|         9500|
|       107| Arjun Verma|     Pune|      2000|        11500|
|       108|  Meera Nair|    Kochi|         0|        11500|
|       110| Nisha Reddy|Bangalore|      2500|        14000|
+----------+------------+---------+----------+-------------+



In [111]:
patient_fees_df.withColumn(
    "next_fee",
    lead("total_fees").over(window_spec)
).show()

+----------+------------+---------+----------+--------+
|patient_id|patient_name|     city|total_fees|next_fee|
+----------+------------+---------+----------+--------+
|       110| Nisha Reddy|Bangalore|      2500|    2500|
|       101|Rahul Sharma|Hyderabad|      2500|    2500|
|       104| Sneha Patel|  Chennai|      2500|    2000|
|       102| Priya Reddy|Bangalore|      2000|    2000|
|       107| Arjun Verma|     Pune|      2000|    1500|
|       103|  Amit Kumar|   Mumbai|      1500|    1000|
|       105|  Farhan Ali|    Delhi|      1000|       0|
|       108|  Meera Nair|    Kochi|         0|    NULL|
+----------+------------+---------+----------+--------+



In [112]:
patient_fees_df.withColumn(
    "previous_fee",
    lag("total_fees").over(window_spec)
).show()

+----------+------------+---------+----------+------------+
|patient_id|patient_name|     city|total_fees|previous_fee|
+----------+------------+---------+----------+------------+
|       110| Nisha Reddy|Bangalore|      2500|        NULL|
|       101|Rahul Sharma|Hyderabad|      2500|        2500|
|       104| Sneha Patel|  Chennai|      2500|        2500|
|       102| Priya Reddy|Bangalore|      2000|        2500|
|       107| Arjun Verma|     Pune|      2000|        2000|
|       103|  Amit Kumar|   Mumbai|      1500|        2000|
|       105|  Farhan Ali|    Delhi|      1000|        1500|
|       108|  Meera Nair|    Kochi|         0|        1000|
+----------+------------+---------+----------+------------+



In [116]:
# PART 8: JSON PROCESSING
%%writefile patient_preferences.json
[
{
"patient_id":101,
"preferred_hospital":"Apollo",
"contact":{
"phone":"9876500011",
"email":"rahul@gmail.com"
}
},
{
"patient_id":102,
"preferred_hospital":"Yashoda",
"contact":{
"phone":null,
"email":"priya@gmail.com"
}
},
{
"patient_id":103,
"preferred_hospital":"Care",
"contact":{
"phone":"9876500013",
"email":null
}
},
{
"patient_id":104,
"preferred_hospital":null,
"contact":{
"phone":"9876500014",
"email":"sneha@gmail.com"
}
}
]

Overwriting patient_preferences.json


In [117]:
preferences_df = spark.read.json("patient_preferences.json")

In [118]:
preferences_df = spark.read.option(
    "multiline",
    "true"
).json(
    "patient_preferences.json"
)

In [119]:
preferences_df.printSchema()

root
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- preferred_hospital: string (nullable = true)



In [120]:
preferences_df.select(
    "patient_id",
    "preferred_hospital",
    col("contact.phone").alias("phone")
).show()

+----------+------------------+----------+
|patient_id|preferred_hospital|     phone|
+----------+------------------+----------+
|       101|            Apollo|9876500011|
|       102|           Yashoda|      NULL|
|       103|              Care|9876500013|
|       104|              NULL|9876500014|
+----------+------------------+----------+



In [121]:
preferences_df.select(
    "patient_id",
    "preferred_hospital",
    col("contact.email").alias("email")
).show()

+----------+------------------+---------------+
|patient_id|preferred_hospital|          email|
+----------+------------------+---------------+
|       101|            Apollo|rahul@gmail.com|
|       102|           Yashoda|priya@gmail.com|
|       103|              Care|           NULL|
|       104|              NULL|sneha@gmail.com|
+----------+------------------+---------------+



In [122]:
preferences_df.filter(
    col("contact.phone").isNull()
).show()

+--------------------+----------+------------------+
|             contact|patient_id|preferred_hospital|
+--------------------+----------+------------------+
|{priya@gmail.com,...|       102|           Yashoda|
+--------------------+----------+------------------+



In [123]:
preferences_df.filter(
    col("contact.email").isNull()
).show()

+------------------+----------+------------------+
|           contact|patient_id|preferred_hospital|
+------------------+----------+------------------+
|{NULL, 9876500013}|       103|              Care|
+------------------+----------+------------------+



In [124]:
preferences_df.filter(
    col("preferred_hospital").isNull()
).show()

+--------------------+----------+------------------+
|             contact|patient_id|preferred_hospital|
+--------------------+----------+------------------+
|{sneha@gmail.com,...|       104|              NULL|
+--------------------+----------+------------------+



In [125]:
preferences_df = preferences_df.withColumn(
    "phone",
    col("contact.phone")
).fillna(
    {"phone": "Not Provided"}
)

In [126]:
preferences_df = preferences_df.withColumn(
    "email",
    col("contact.email")
).fillna(
    {"email": "Not Provided"}
)

In [127]:
patients_df.join(
    preferences_df,
    "patient_id",
    "left"
).show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+--------------------+------------------+------------+---------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|             contact|preferred_hospital|       phone|          email|
+----------+------------+---------+---+------+-------------+----------------+-------------------+--------------------+------------------+------------+---------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|{rahul@gmail.com,...|            Apollo|  9876500011|rahul@gmail.com|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|{priya@gmail.com,...|           Yashoda|Not Provided|priya@gmail.com|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|  {NULL, 9876500013}|              Care|  9876500013|   Not Provided

In [128]:
# PART 9 : SPARK SQL
patients_df.createOrReplaceTempView("patients")

In [129]:
appointments_df.createOrReplaceTempView("appointments")

In [130]:
spark.sql("""
SELECT *
FROM patients
""").show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|
+----------+------------+---------+---+------+-------------+----------------+-------------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|           Complete|
|       106|  Neha Singh|     NULL| 38|Female|           A+|        Inactive|         Incomplete|
|       107| Arjun Verma|     Pune| 26|  Male|           B+|          Active|           Complete|
|       108|  Meera 

In [131]:
spark.sql("""
SELECT *
FROM patients
WHERE city = 'Hyderabad'
""").show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|
+----------+------------+---------+---+------+-------------+----------------+-------------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|
|       109|   Kiran Rao|Hyderabad| 33|  Male|Not Available|        Inactive|           Complete|
+----------+------------+---------+---+------+-------------+----------------+-------------------+



In [132]:
spark.sql("""
SELECT city, COUNT(*) AS patient_count
FROM patients
GROUP BY city
""").show()

+---------+-------------+
|     city|patient_count|
+---------+-------------+
|Bangalore|            2|
|    Kochi|            1|
|  Chennai|            1|
|     NULL|            1|
|   Mumbai|            1|
|     Pune|            1|
|    Delhi|            1|
|Hyderabad|            2|
+---------+-------------+



In [133]:
spark.sql("""
SELECT department, COUNT(*) AS appointment_count
FROM appointments
GROUP BY department
""").show()

+-----------+-----------------+
| department|appointment_count|
+-----------+-----------------+
|  Neurology|                2|
|Dermatology|                3|
| Cardiology|                3|
|Orthopedics|                2|
+-----------+-----------------+



In [134]:
spark.sql("""
SELECT department, AVG(consultation_fee) AS average_fee
FROM appointments
GROUP BY department
""").show()

+-----------+-----------------+
| department|      average_fee|
+-----------+-----------------+
|  Neurology|           2000.0|
|Dermatology|666.6666666666666|
| Cardiology|           1500.0|
|Orthopedics|           2500.0|
+-----------+-----------------+



In [135]:
spark.sql("""
SELECT MAX(consultation_fee) AS highest_fee
FROM appointments
""").show()

+-----------+
|highest_fee|
+-----------+
|       2500|
+-----------+



In [136]:
spark.sql("""
SELECT p.patient_name, COUNT(a.appointment_id) AS appointment_count
FROM patients p
JOIN appointments a
ON p.patient_id = a.patient_id
GROUP BY p.patient_name
""").show()

+------------+-----------------+
|patient_name|appointment_count|
+------------+-----------------+
|  Amit Kumar|                1|
|Rahul Sharma|                2|
|  Meera Nair|                1|
| Sneha Patel|                1|
|  Farhan Ali|                1|
| Arjun Verma|                1|
| Nisha Reddy|                1|
| Priya Reddy|                1|
+------------+-----------------+



In [137]:
spark.sql("""
SELECT p.patient_name, SUM(a.consultation_fee) AS total_spent
FROM patients p
JOIN appointments a
ON p.patient_id = a.patient_id
GROUP BY p.patient_name
ORDER BY total_spent DESC
LIMIT 5
""").show()

+------------+-----------+
|patient_name|total_spent|
+------------+-----------+
|Rahul Sharma|       2500|
| Sneha Patel|       2500|
| Nisha Reddy|       2500|
| Arjun Verma|       2000|
| Priya Reddy|       2000|
+------------+-----------+



In [138]:
# PART 10: ETL PROJECT
patients_df = spark.read.csv("patients.csv", header=True, inferSchema=True)

appointments_df = spark.read.csv("appointments.csv", header=True, inferSchema=True)

In [139]:
preferences_df = spark.read.option("multiline", "true").json("patient_preferences.json")

In [140]:
patients_df = patients_df.fillna({
    "city": "Unknown",
    "blood_group": "Not Available"
})

appointments_df = appointments_df.fillna({
    "consultation_fee": 0
})

preferences_df = preferences_df.withColumn(
    "phone",
    col("contact.phone")
).withColumn(
    "email",
    col("contact.email")
).fillna({
    "phone": "Not Provided",
    "email": "Not Provided",
    "preferred_hospital": "Not Selected"
})

In [141]:
final_df = patients_df.join(
    appointments_df,
    "patient_id",
    "left"
).join(
    preferences_df,
    "patient_id",
    "left"
)

In [142]:
final_df = final_df.withColumn(
    "age_group",
    when(col("age") < 30, "Young")
    .when(col("age") < 50, "Adult")
    .otherwise("Senior")
)

In [143]:
revenue_metrics_df = appointments_df.agg(
    sum("consultation_fee").alias("total_revenue"),
    avg("consultation_fee").alias("average_fee"),
    max("consultation_fee").alias("highest_fee"),
    min("consultation_fee").alias("lowest_fee")
)

revenue_metrics_df.show()

+-------------+-----------+-----------+----------+
|total_revenue|average_fee|highest_fee|lowest_fee|
+-------------+-----------+-----------+----------+
|        15500|     1550.0|       2500|         0|
+-------------+-----------+-----------+----------+



In [144]:
patient_spending_df = final_df.groupBy(
    "patient_id",
    "patient_name"
).agg(
    sum("consultation_fee").alias("total_spent")
)

patient_spending_df.show()

+----------+------------+-----------+
|patient_id|patient_name|total_spent|
+----------+------------+-----------+
|       107| Arjun Verma|       2000|
|       108|  Meera Nair|          0|
|       109|   Kiran Rao|       NULL|
|       110| Nisha Reddy|       2500|
|       105|  Farhan Ali|       1000|
|       101|Rahul Sharma|       2500|
|       104| Sneha Patel|       2500|
|       103|  Amit Kumar|       1500|
|       106|  Neha Singh|       NULL|
|       102| Priya Reddy|       2000|
+----------+------------+-----------+



In [145]:
department_revenue_df = appointments_df.groupBy(
    "department"
).agg(
    sum("consultation_fee").alias("department_revenue")
)

department_revenue_df.show()

+-----------+------------------+
| department|department_revenue|
+-----------+------------------+
|  Neurology|              4000|
|Dermatology|              2000|
| Cardiology|              4500|
|Orthopedics|              5000|
+-----------+------------------+



In [146]:
final_df.write.mode("overwrite").parquet("hospital_final_output")

In [147]:
print("Hospital Analytics Report")

print("Total Patients:", patients_df.count())

print("Total Appointments:", appointments_df.count())

revenue_metrics_df.show()

patient_spending_df.show()

department_revenue_df.show()

final_df.show()

Hospital Analytics Report
Total Patients: 10
Total Appointments: 10
+-------------+-----------+-----------+----------+
|total_revenue|average_fee|highest_fee|lowest_fee|
+-------------+-----------+-----------+----------+
|        15500|     1550.0|       2500|         0|
+-------------+-----------+-----------+----------+

+----------+------------+-----------+
|patient_id|patient_name|total_spent|
+----------+------------+-----------+
|       107| Arjun Verma|       2000|
|       108|  Meera Nair|          0|
|       109|   Kiran Rao|       NULL|
|       110| Nisha Reddy|       2500|
|       105|  Farhan Ali|       1000|
|       101|Rahul Sharma|       2500|
|       104| Sneha Patel|       2500|
|       103|  Amit Kumar|       1500|
|       106|  Neha Singh|       NULL|
|       102| Priya Reddy|       2000|
+----------+------------+-----------+

+-----------+------------------+
| department|department_revenue|
+-----------+------------------+
|  Neurology|              4000|
|Dermatolog